In [ ]:
!pip -q install -U transformers accelerate scikit-learn openpyxl

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Any, Dict
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score, classification_report
import transformers

set_seed(42)
print("Transformers version:", transformers.__version__)

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

TRAIN_PATH = "train.xlsx"
VALID_PATH = "valid.xlsx"
TEST_PATH  = "test.xlsx"

TEXT_COL = "clean_text"
LABEL_DIALECT_COL = "dialect"
LABEL_HATE_COL    = "hate"
LABEL_TOPIC_COL   = "topic"

MAX_LENGTH = 128
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 3
OUTPUT_DIR = "./camelbert_mix_multilabel_with_emoji_feature"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

id2dialect = {1: "Saudi", 0: "Egyptian"}
id2hate    = {1: "Hate", 0: "Not Hate"}
id2topic   = {1: "Religious", 0: "Political"}

HATE_EMOJIS = set([
    '💦', '🐖', '🐷', '🐽', '👞', '🐕', '🐶', '💩', '🐄', '🐮',
    '🐑', '🐏', '👎', '😡', '🤬', '👺', '👿', '😠'
])

def emoji_flag(text: str) -> int:
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

def load_excel(path: str) -> pd.DataFrame:
    return pd.read_excel(path, engine="openpyxl")

def load_and_validate_xlsx(path: str) -> pd.DataFrame:
    df = load_excel(path)

    required = {TEXT_COL, LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}\nAvailable columns: {list(df.columns)}")

    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

    for c in [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]:
        if df[c].isna().any():
            bad_rows = df[df[c].isna()].head(5)
            raise ValueError(f"{path}: column '{c}' has NaN values. Examples:\n{bad_rows[[TEXT_COL, c]]}")

        df[c] = pd.to_numeric(df[c], errors="raise").astype(int)
        bad = ~df[c].isin([0, 1])
        if bad.any():
            examples = df.loc[bad, [TEXT_COL, c]].head(5)
            raise ValueError(f"{path}: column '{c}' must be 0/1 only. Examples:\n{examples}")

    return df

train_df = load_and_validate_xlsx(TRAIN_PATH)
valid_df = load_and_validate_xlsx(VALID_PATH)
test_df  = load_and_validate_xlsx(TEST_PATH)

print("Rows:", len(train_df), len(valid_df), len(test_df))
print("Train label counts:")
print("dialect:", train_df[LABEL_DIALECT_COL].value_counts().to_dict())
print("hate:",    train_df[LABEL_HATE_COL].value_counts().to_dict())
print("topic:",   train_df[LABEL_TOPIC_COL].value_counts().to_dict())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiLabelDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        text = row[TEXT_COL]

        enc = self.tok(text, truncation=True, padding=False, max_length=self.max_len)

        labels = np.array([
            int(row[LABEL_DIALECT_COL]),
            int(row[LABEL_HATE_COL]),
            int(row[LABEL_TOPIC_COL]),
        ], dtype=np.float32)

        item = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": labels,
            "emoji_flag": float(emoji_flag(text)),
        }

        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"]

        return item

@dataclass
class DataCollatorMultiLabel:
    tokenizer: Any

    def __call__(self, features):
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)
        eflag  = torch.tensor([f["emoji_flag"] for f in features], dtype=torch.float32)

        for f in features:
            f.pop("labels")
            f.pop("emoji_flag")

        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["labels"] = labels
        batch["emoji_flag"] = eflag
        return batch

train_ds = MultiLabelDataset(train_df, tokenizer, MAX_LENGTH)
valid_ds = MultiLabelDataset(valid_df, tokenizer, MAX_LENGTH)
test_ds  = MultiLabelDataset(test_df, tokenizer, MAX_LENGTH)
collator = DataCollatorMultiLabel(tokenizer)

class MultiLabelCAMeLBERT(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 1, 3)
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.emoji_boost = nn.Parameter(torch.tensor(0.75, dtype=torch.float32))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, emoji_flag=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None,
        )

        pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0, :]
        pooled = self.drop(pooled)

        if emoji_flag is None:
            emoji_flag = torch.zeros(pooled.size(0), device=pooled.device, dtype=pooled.dtype)
        else:
            emoji_flag = emoji_flag.to(pooled.dtype)

        x = torch.cat([pooled, emoji_flag.unsqueeze(1)], dim=1)
        logits = self.classifier(x)
        logits[:, 1] = logits[:, 1] + self.emoji_boost * emoji_flag

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = MultiLabelCAMeLBERT(MODEL_NAME).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    metrics = {}
    metrics["overall_acc"] = np.mean(np.all(labels == preds, axis=1))

    metrics["dialect_acc"] = accuracy_score(labels[:, 0], preds[:, 0])
    metrics["dialect_f1"] = f1_score(labels[:, 0], preds[:, 0], average="macro", zero_division=0)
    metrics["dialect_f1_micro"] = f1_score(labels[:, 0], preds[:, 0], average="micro", zero_division=0)

    metrics["hate_acc"] = accuracy_score(labels[:, 1], preds[:, 1])
    metrics["hate_f1"] = f1_score(labels[:, 1], preds[:, 1], average="macro", zero_division=0)
    metrics["hate_f1_micro"] = f1_score(labels[:, 1], preds[:, 1], average="micro", zero_division=0)

    metrics["topic_acc"] = accuracy_score(labels[:, 2], preds[:, 2])
    metrics["topic_f1"] = f1_score(labels[:, 2], preds[:, 2], average="macro", zero_division=0)
    metrics["topic_f1_micro"] = f1_score(labels[:, 2], preds[:, 2], average="micro", zero_division=0)

    metrics["avg_f1"] = (metrics["dialect_f1"] + metrics["hate_f1"] + metrics["topic_f1"]) / 3.0
    metrics["avg_f1_micro"] = (metrics["dialect_f1_micro"] + metrics["hate_f1_micro"] + metrics["topic_f1_micro"]) / 3.0

    return metrics

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

pred_out = trainer.predict(test_ds)
logits = np.asarray(pred_out.predictions)
labels = np.asarray(pred_out.label_ids)

probs = 1.0 / (1.0 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

# ================== SAVE TEST PREDICTIONS TO EXCEL ==================
test_results_df = test_df.copy()

test_results_df["true_dialect"] = labels[:, 0].astype(int)
test_results_df["pred_dialect"] = preds[:, 0].astype(int)
test_results_df["true_dialect_label"] = test_results_df["true_dialect"].map(id2dialect)
test_results_df["pred_dialect_label"] = test_results_df["pred_dialect"].map(id2dialect)
test_results_df["dialect_correct"] = test_results_df["true_dialect"] == test_results_df["pred_dialect"]

test_results_df["true_hate"] = labels[:, 1].astype(int)
test_results_df["pred_hate"] = preds[:, 1].astype(int)
test_results_df["true_hate_label"] = test_results_df["true_hate"].map(id2hate)
test_results_df["pred_hate_label"] = test_results_df["pred_hate"].map(id2hate)
test_results_df["hate_correct"] = test_results_df["true_hate"] == test_results_df["pred_hate"]

test_results_df["true_topic"] = labels[:, 2].astype(int)
test_results_df["pred_topic"] = preds[:, 2].astype(int)
test_results_df["true_topic_label"] = test_results_df["true_topic"].map(id2topic)
test_results_df["pred_topic_label"] = test_results_df["pred_topic"].map(id2topic)
test_results_df["topic_correct"] = test_results_df["true_topic"] == test_results_df["pred_topic"]

test_results_df["dialect_prob_egyptian"] = 1 - probs[:, 0]
test_results_df["dialect_prob_saudi"] = probs[:, 0]

test_results_df["hate_prob_not_hate"] = 1 - probs[:, 1]
test_results_df["hate_prob_hate"] = probs[:, 1]

test_results_df["topic_prob_political"] = 1 - probs[:, 2]
test_results_df["topic_prob_religious"] = probs[:, 2]

test_results_df["all_correct"] = (
    test_results_df["dialect_correct"] &
    test_results_df["hate_correct"] &
    test_results_df["topic_correct"]
)

test_results_df.to_excel("test_predictions.xlsx", index=False, engine="openpyxl")

print("\nTest predictions saved to: test_predictions.xlsx")
# ====================================================================

overall_acc = np.mean(np.all(labels == preds, axis=1))
print(f"\nOverall exact-match accuracy (all labels together): {overall_acc:.4f}")

def print_report_with_micro(y_true, y_pred, title: str):
    rep = classification_report(y_true, y_pred, digits=4, zero_division=0, output_dict=True)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    def row(k):
        return rep.get(k, {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})

    r0 = row("0")
    r1 = row("1")
    rmacro = row("macro avg")
    rweight = row("weighted avg")
    total_support = int(r0["support"]) + int(r1["support"])

    print(f"\n--- {title} ---")
    print(f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'f1-micro':>10}{'support':>10}")
    print(f"{'0.0':>14}{r0['precision']:>10.4f}{r0['recall']:>10.4f}{r0['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r0['support']):>10}")
    print(f"{'1.0':>14}{r1['precision']:>10.4f}{r1['recall']:>10.4f}{r1['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r1['support']):>10}")
    print()
    print(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.4f}{f1_micro:>10.4f}{total_support:>10}")
    print(f"{'macro avg':>14}{rmacro['precision']:>10.4f}{rmacro['recall']:>10.4f}{rmacro['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rmacro['support']):>10}")
    print(f"{'weighted avg':>14}{rweight['precision']:>10.4f}{rweight['recall']:>10.4f}{rweight['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rweight['support']):>10}")

print_report_with_micro(labels[:, 0].astype(int), preds[:, 0].astype(int), "Dialect report (1=saudi,0=egyptian)")
print_report_with_micro(labels[:, 1].astype(int), preds[:, 1].astype(int), "Hate report (1=hate,0=not hate)")
print_report_with_micro(labels[:, 2].astype(int), preds[:, 2].astype(int), "Topic report (1=religious,0=political)")

os.makedirs(OUTPUT_DIR, exist_ok=True)
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pytorch_model.bin"))
print("\nSaved to:", OUTPUT_DIR)
print("Learned emoji_boost (final):", float(model.emoji_boost.detach().cpu().item()))

def predict(text: str, threshold: float = 0.5):
    model.eval()

    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    eflag = torch.tensor([float(emoji_flag(text))], device=device)

    with torch.no_grad():
        out = model(**enc, emoji_flag=eflag)

    probs = torch.sigmoid(out["logits"]).squeeze(0)
    pred = (probs >= threshold).long()

    dialect_id = int(pred[0].item())
    hate_id = int(pred[1].item())
    topic_id = int(pred[2].item())

    dialect_p = float(probs[0].item())
    hate_p = float(probs[1].item())
    topic_p = float(probs[2].item())

    return {
        "dialect_id": dialect_id,
        "hate_id": hate_id,
        "topic_id": topic_id,
        "dialect_label": id2dialect[dialect_id],
        "hate_label": id2hate[hate_id],
        "topic_label": id2topic[topic_id],
        "dialect_probs": {"Egyptian": 1 - dialect_p, "Saudi": dialect_p},
        "hate_probs": {"Not Hate": 1 - hate_p, "Hate": hate_p},
        "topic_probs": {"Political": 1 - topic_p, "Religious": topic_p},
        "dialect_confidence": dialect_p if dialect_id == 1 else (1 - dialect_p),
        "hate_confidence": hate_p if hate_id == 1 else (1 - hate_p),
        "topic_confidence": topic_p if topic_id == 1 else (1 - topic_p),
        "emoji_flag": int(emoji_flag(text)),
    }

print("\n=== Interactive Mode ===")
print("Type: exit / quit / stop to end\n")

while True:
    text = input("Tweet> ").strip()
    if text.lower() in {"exit", "quit", "stop"}:
        print("Done.")
        break
    if not text:
        continue

    result = predict(text, threshold=0.5)

    print(
        f"Dialect: {result['dialect_label']} "
        f"(confidence={result['dialect_confidence']:.3f}, Egyptian={result['dialect_probs']['Egyptian']:.3f}, Saudi={result['dialect_probs']['Saudi']:.3f})"
    )
    print(
        f"Hate: {result['hate_label']} "
        f"(confidence={result['hate_confidence']:.3f}, Not Hate={result['hate_probs']['Not Hate']:.3f}, Hate={result['hate_probs']['Hate']:.3f}, emoji_flag={result['emoji_flag']})"
    )
    print(
        f"Topic: {result['topic_label']} "
        f"(confidence={result['topic_confidence']:.3f}, Political={result['topic_probs']['Political']:.3f}, Religious={result['topic_probs']['Religious']:.3f})"
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 79.7 MB/s eta 0:00:00
Transformers version: 5.7.0
Device: cuda
Rows: 2773 595 594
Train label counts:
dialect: {1: 1417, 0: 1356}
hate: {1: 1443, 0: 1330}
topic: {0: 1407, 1: 1366}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

[transformers] BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

/tmp/ipykernel_2104/3632277148.py:127: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)


Epoch,Training Loss,Validation Loss,Overall Acc,Dialect Acc,Dialect F1,Dialect F1 Micro,Hate Acc,Hate F1,Hate F1 Micro,Topic Acc,Topic F1,Topic F1 Micro,Avg F1,Avg F1 Micro
1,0.336924,0.210094,0.752941,0.926050,0.925920,0.926050,0.818487,0.818487,0.818487,0.991597,0.991596,0.991597,0.912001,0.912045
2,0.166454,0.202188,0.759664,0.926050,0.926025,0.926050,0.835294,0.835003,0.835294,0.986555,0.986552,0.986555,0.915860,0.915966
3,0.103582,0.204286,0.756303,0.921008,0.921008,0.921008,0.830252,0.830204,0.830252,0.988235,0.988234,0.988235,0.913149,0.913165



Test predictions saved to: test_predictions.xlsx

Overall exact-match accuracy (all labels together): 0.7593

--- Dialect report (1=saudi,0=egyptian) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.9638    0.9214    0.9421    0.9394       318
           1.0    0.9138    0.9601    0.9364    0.9394       276

      accuracy                        0.9394    0.9394       594
     macro avg    0.9388    0.9408    0.9393    0.9394       594
  weighted avg    0.9406    0.9394    0.9395    0.9394       594

--- Hate report (1=hate,0=not hate) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.8118    0.8034    0.8076    0.8131       290
           1.0    0.8143    0.8224    0.8183    0.8131       304

      accuracy                        0.8131    0.8131       594
     macro avg    0.8131    0.8129    0.8130    0.8131       594
  weighted avg    0.8131    0.8131    0.8131    0.8131       594

--- Topic report (1=r

# **LIME**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MODEL_DIR = "/content/drive/MyDrive/models/camelbert_model"
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class MultiLabelCAMeLBERT(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 1, 3)
        self.emoji_boost = nn.Parameter(torch.tensor(0.75, dtype=torch.float32))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, emoji_flag=None):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None,
        )

        pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0, :]
        pooled = self.drop(pooled)

        if emoji_flag is None:
            emoji_flag = torch.zeros(pooled.size(0), device=pooled.device, dtype=pooled.dtype)
        else:
            emoji_flag = emoji_flag.to(pooled.dtype)

        x = torch.cat([pooled, emoji_flag.unsqueeze(1)], dim=1)
        logits = self.classifier(x)

        # تأثير الإيموجي على الكراهية فقط
        logits[:, 1] = logits[:, 1] + self.emoji_boost * emoji_flag

        return {"logits": logits}

In [ ]:
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiLabelCAMeLBERT(MODEL_NAME).to(device)

model.load_state_dict(
    torch.load(
        os.path.join(MODEL_DIR, "pytorch_model.bin"),
        map_location=device
    )
)

model.eval()

print("Model loaded successfully ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Model loaded successfully ✅


In [ ]:
HATE_EMOJIS = set([
    '💦','🐖','🐷','🐽','👞','🐕','🐶','💩',
    '🐄','🐮','🐑','🐏','👎','😡','🤬','👺','👿','😠'
])

def emoji_flag(text):
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

In [ ]:
import numpy as np

MAX_LENGTH = 128

tasks = {
    0: ["Egyptian", "Saudi"],
    1: ["Not Hate", "Hate"],
    2: ["Political", "Religious"]
}

In [ ]:
def predict_raw(text):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    eflag = torch.tensor([float(emoji_flag(text))], device=device)

    with torch.no_grad():
        out = model(**enc, emoji_flag=eflag)

    probs = torch.sigmoid(out["logits"]).squeeze(0).cpu().numpy()

    return {
        "Dialect": {
            "Prediction": "Saudi" if probs[0] >= 0.5 else "Egyptian",
            "Egyptian": 1 - probs[0],
            "Saudi": probs[0]
        },
        "Hate": {
            "Prediction": "Hate" if probs[1] >= 0.5 else "Not Hate",
            "Not Hate": 1 - probs[1],
            "Hate": probs[1]
        },
        "Topic": {
            "Prediction": "Religious" if probs[2] >= 0.5 else "Political",
            "Political": 1 - probs[2],
            "Religious": probs[2]
        },
        "emoji_flag": int(emoji_flag(text))
    }

In [ ]:
def predict_proba_label(label_index):
    def classifier_fn(texts):
        model.eval()
        outputs = []

        for text in texts:
            enc = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_LENGTH
            ).to(device)

            eflag = torch.tensor([float(emoji_flag(text))], device=device)

            with torch.no_grad():
                out = model(**enc, emoji_flag=eflag)

            probs = torch.sigmoid(out["logits"]).squeeze(0).cpu().numpy()

            p = probs[label_index]
            outputs.append([1 - p, p])

        return np.array(outputs)

    return classifier_fn

In [ ]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=00c606f3d44887f3c21211842a6d4be70d6657596631e0fbf2491764489d585f
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


# **Examples from the Test Set**

In [ ]:
from lime.lime_text import LimeTextExplainer
import pandas as pd

# =========================
# Live Explain Function
# =========================
def explain_text_live(text):
    pred = predict_raw(text)

    print("\n" + "="*50)
    print(f"Input text: {text}")
    print("="*50)

    print("\nPrediction:")
    print(f"- Dialect: {pred['Dialect']['Prediction']} "
          f"(Egyptian={pred['Dialect']['Egyptian']:.2f}, Saudi={pred['Dialect']['Saudi']:.2f})")

    print(f"- Hate: {pred['Hate']['Prediction']} "
          f"(Not Hate={pred['Hate']['Not Hate']:.2f}, Hate={pred['Hate']['Hate']:.2f})")

    print(f"- Topic: {pred['Topic']['Prediction']} "
          f"(Political={pred['Topic']['Political']:.2f}, Religious={pred['Topic']['Religious']:.2f})")

    print(f"- Emoji flag: {pred['emoji_flag']}")

    print("\nBrief numeric explanation:")

    # تحديد الفئة المختارة لكل مهمة
    selected_classes = {
        0: 1 if pred["Dialect"]["Prediction"] == "Saudi" else 0,
        1: 1 if pred["Hate"]["Prediction"] == "Hate" else 0,
        2: 1 if pred["Topic"]["Prediction"] == "Religious" else 0
    }

    task_names = {
        0: "Dialect",
        1: "Hate",
        2: "Topic"
    }

    # تفسير كل مهمة
    for label_index, class_pair in tasks.items():
        class_id = selected_classes[label_index]
        class_name = class_pair[class_id]

        explainer = LimeTextExplainer(
            class_names=class_pair,
            split_expression=r"\s+",
            random_state=42   # مهم عشان النتائج ثابتة
        )

        exp = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predict_proba_label(label_index),
            labels=[class_id],
            num_features=5
        )

        df = pd.DataFrame(
            exp.as_list(label=class_id),
            columns=["Feature", "Weight"]
        )

        df["Abs"] = df["Weight"].abs()
        df = df.sort_values("Abs", ascending=False).drop(columns="Abs").head(3)

        print(f"\n{task_names[label_index]} → {class_name}")

        for _, row in df.iterrows():
            direction = "supports" if row["Weight"] > 0 else "opposes"
            print(f"  - {row['Feature']}: {row['Weight']:.4f} ({direction})")


# =========================
# Live Mode
# =========================
print("=== Live Explainable Mode ===")
print("Type a sentence, then press Enter")
print("To exit, type: exit / quit / stop\n")

while True:
    text = input("Tweet> ").strip()

    if text.lower() in ["exit", "quit", "stop"]:
        print("Done.")
        break

    if text == "":
        continue

    explain_text_live(text)

=== Live Explainable Mode ===
Type a sentence, then press Enter
To exit, type: exit / quit / stop

Tweet> انت ماخذ الموضوع ببساطه شديده السياسه يا مشروعي يا انك عدوي وانت تعرف ان السعوديه اليوم مشروعها مختلف المحور الثلاثي لذالك حنا اعداء بالنسبه لهم الزمن تكتشف الشي 

Input text: انت ماخذ الموضوع ببساطه شديده السياسه يا مشروعي يا انك عدوي وانت تعرف ان السعوديه اليوم مشروعها مختلف المحور الثلاثي لذالك حنا اعداء بالنسبه لهم الزمن تكتشف الشي

Prediction:
- Dialect: Saudi (Egyptian=0.00, Saudi=1.00)
- Hate: Hate (Not Hate=0.35, Hate=0.65)
- Topic: Political (Political=0.99, Religious=0.01)
- Emoji flag: 0

Brief numeric explanation:

Dialect → Saudi
  - السعوديه: 0.0763 (supports)
  - حنا: 0.0625 (supports)
  - ماخذ: 0.0533 (supports)

Hate → Hate
  - عدوي: 0.4457 (supports)
  - يا: 0.2720 (supports)
  - السياسه: -0.1567 (opposes)

Topic → Political
  - السعوديه: 0.6078 (supports)
  - حنا: 0.1151 (supports)
  - السياسه: 0.0944 (supports)
Tweet> المره دي الشعب اللي هيفشخهم احنا لسه بندفع ف

# **External Examples for Evaluating Model Performance**

In [ ]:
from lime.lime_text import LimeTextExplainer
import pandas as pd

# =========================
# Live Explain Function
# =========================
def explain_text_live(text):
    pred = predict_raw(text)

    print("\n" + "="*50)
    print(f"Input text: {text}")
    print("="*50)

    print("\nPrediction:")
    print(f"- Dialect: {pred['Dialect']['Prediction']} "
          f"(Egyptian={pred['Dialect']['Egyptian']:.2f}, Saudi={pred['Dialect']['Saudi']:.2f})")

    print(f"- Hate: {pred['Hate']['Prediction']} "
          f"(Not Hate={pred['Hate']['Not Hate']:.2f}, Hate={pred['Hate']['Hate']:.2f})")

    print(f"- Topic: {pred['Topic']['Prediction']} "
          f"(Political={pred['Topic']['Political']:.2f}, Religious={pred['Topic']['Religious']:.2f})")

    print(f"- Emoji flag: {pred['emoji_flag']}")

    print("\nBrief numeric explanation:")

    # تحديد الفئة المختارة لكل مهمة
    selected_classes = {
        0: 1 if pred["Dialect"]["Prediction"] == "Saudi" else 0,
        1: 1 if pred["Hate"]["Prediction"] == "Hate" else 0,
        2: 1 if pred["Topic"]["Prediction"] == "Religious" else 0
    }

    task_names = {
        0: "Dialect",
        1: "Hate",
        2: "Topic"
    }

    # تفسير كل مهمة
    for label_index, class_pair in tasks.items():
        class_id = selected_classes[label_index]
        class_name = class_pair[class_id]

        explainer = LimeTextExplainer(
            class_names=class_pair,
            split_expression=r"\s+",
            random_state=42   # مهم عشان النتائج ثابتة
        )

        exp = explainer.explain_instance(
            text_instance=text,
            classifier_fn=predict_proba_label(label_index),
            labels=[class_id],
            num_features=5
        )

        df = pd.DataFrame(
            exp.as_list(label=class_id),
            columns=["Feature", "Weight"]
        )

        df["Abs"] = df["Weight"].abs()
        df = df.sort_values("Abs", ascending=False).drop(columns="Abs").head(3)

        print(f"\n{task_names[label_index]} → {class_name}")

        for _, row in df.iterrows():
            direction = "supports" if row["Weight"] > 0 else "opposes"
            print(f"  - {row['Feature']}: {row['Weight']:.4f} ({direction})")


# =========================
# Live Mode
# =========================
print("=== Live Explainable Mode ===")
print("Type a sentence, then press Enter")
print("To exit, type: exit / quit / stop\n")

while True:
    text = input("Tweet> ").strip()

    if text.lower() in ["exit", "quit", "stop"]:
        print("Done.")
        break

    if text == "":
        continue

    explain_text_live(text)

=== Live Explainable Mode ===
Type a sentence, then press Enter
To exit, type: exit / quit / stop

Tweet> وربنا اللي بيحصل ده مش طبيعي خالص واللي بيحصل في البلد محتاج وقفة جد 😡

Input text: وربنا اللي بيحصل ده مش طبيعي خالص واللي بيحصل في البلد محتاج وقفة جد 😡

Prediction:
- Dialect: Egyptian (Egyptian=0.98, Saudi=0.02)
- Hate: Not Hate (Not Hate=0.97, Hate=0.03)
- Topic: Political (Political=0.95, Religious=0.05)
- Emoji flag: 1

Brief numeric explanation:

Dialect → Egyptian
  - ده: 0.0091 (supports)
  - خالص: 0.0089 (supports)
  - مش: 0.0083 (supports)

Hate → Not Hate
  - بيحصل: 0.0230 (supports)
  - 😡: -0.0207 (opposes)
  - وربنا: 0.0205 (supports)

Topic → Political
  - البلد: 0.6621 (supports)
  - وربنا: -0.0690 (opposes)
  - بيحصل: 0.0641 (supports)
Tweet> يا أذناب المحور الثلاثي تظنون إنكم بتهزون جبل حنا أدرى بخبثكم يا 🐕 تلبسون ثوب الدين وأنتم 👿 في هيئة بشر السعودية خط أحمر والزمن بيكشف خيانتكم يا عبيد الصهاينة

Input text: يا أذناب المحور الثلاثي تظنون إنكم بتهزون جبل حنا أدر